# Fine-tune LayoutLMv3 for FDR field extraction (DepositTrack, pass A)

Run this on a **GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU; or Kaggle with GPU on).

What it does:
1. Installs the synthetic FDR generator (`ml/synth`) from the DepositTrack repository and renders labelled FDR images.
2. Fine-tunes `microsoft/layoutlmv3-base` for token classification (BIO labels for 11 fields), starting from the paper's FUNSD settings (learning rate 1e-5, batch 16 — Huang et al., 2022).
3. Reports entity-level F1 on held-out synthetic pages (**synthetic data — label it as such**).
4. Saves the model as `layoutlmv3-fdr.zip`. Download it and unzip into `models/layoutlmv3-fdr/` at the repository root; the API picks it up automatically (`GET /api/v1/system/status` shows `layoutlm_available: true`).

The API also needs `transformers` and `torch` installed in `apps/api/.venv` to run pass A on CPU.

In [ ]:
# Settings - change only if needed
REPO = "https://github.com/mayank-bajaj-ai24/DepositTrack-AICTE-FDR-Management.git"
N_TRAIN, N_TEST = 2000, 300
EPOCHS = 4
LEARNING_RATE = 1e-5   # LayoutLMv3 paper, FUNSD
BATCH_SIZE = 16        # lower to 8 if the GPU runs out of memory
SEED = 7

In [ ]:
!nvidia-smi
!pip -q install "git+$REPO#subdirectory=ml/synth" transformers datasets seqeval accelerate

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "fdrsynth", "--count", str(N_TRAIN), "--out", "data/train", "--seed", str(SEED)], check=True)
subprocess.run([sys.executable, "-m", "fdrsynth", "--count", str(N_TEST), "--out", "data/test", "--seed", str(SEED + 1000)], check=True)

In [ ]:
import json, glob
from PIL import Image

FIELD_LABELS = ["FDR_NO", "BANK", "BRANCH", "IFSC", "AMOUNT", "AMOUNT_WORDS",
                "ISSUE_DATE", "MATURITY_DATE", "BENEFICIARY", "DEPOSITOR", "LIEN_TEXT"]
LABELS = ["O"] + [f"{p}-{f}" for f in FIELD_LABELS for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def load(split_dir):
    rows = []
    for path in sorted(glob.glob(f"{split_dir}/*.json")):
        rec = json.load(open(path))
        w, h = rec["width"], rec["height"]
        words, boxes, labels, prev = [], [], [], "O"
        for word in rec["words"]:
            x0, y0, x1, y1 = word["box"]
            words.append(word["text"])
            boxes.append([int(1000 * x0 / w), int(1000 * y0 / h), int(1000 * x1 / w), int(1000 * y1 / h)])
            lab = word["label"]
            labels.append("O" if lab == "O" else ("I-" if lab == prev else "B-") + lab)
            prev = lab
        rows.append({"image": f"{split_dir}/{rec['image']}", "words": words, "boxes": boxes,
                     "labels": [label2id[l] for l in labels]})
    return rows

train_rows, test_rows = load("data/train"), load("data/test")
print(len(train_rows), "train pages,", len(test_rows), "test pages")

In [ ]:
from transformers import AutoProcessor
from datasets import Dataset

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

def encode(batch):
    images = [Image.open(p).convert("RGB") for p in batch["image"]]
    return processor(images, batch["words"], boxes=batch["boxes"], word_labels=batch["labels"],
                     truncation=True, padding="max_length", max_length=512)

train_ds = Dataset.from_list(train_rows).map(encode, batched=True, remove_columns=["image", "words", "boxes", "labels"])
test_ds = Dataset.from_list(test_rows).map(encode, batched=True, remove_columns=["image", "words", "boxes", "labels"])
train_ds.set_format("torch"); test_ds.set_format("torch")

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def metrics(pred):
    logits, gold = pred
    predicted = logits.argmax(-1)
    true_seq, pred_seq = [], []
    for p_row, g_row in zip(predicted, gold):
        t, q = [], []
        for p, g in zip(p_row, g_row):
            if g == -100:
                continue
            t.append(id2label[int(g)]); q.append(id2label[int(p)])
        true_seq.append(t); pred_seq.append(q)
    return {"entity_f1": f1_score(true_seq, pred_seq)}

args = TrainingArguments(
    output_dir="out", num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch", save_strategy="no", logging_steps=50, fp16=True, seed=SEED,
    report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=metrics)
trainer.train()
result = trainer.evaluate()
print(result)

In [ ]:
# Per-field report on the synthetic test pages
pred = trainer.predict(test_ds)
predicted = pred.predictions.argmax(-1)
true_seq, pred_seq = [], []
for p_row, g_row in zip(predicted, pred.label_ids):
    pairs = [(id2label[int(g)], id2label[int(p)]) for p, g in zip(p_row, g_row) if g != -100]
    true_seq.append([g for g, _ in pairs]); pred_seq.append([p for _, p in pairs])
report = classification_report(true_seq, pred_seq, digits=4)
print(report)

In [ ]:
import shutil, json
trainer.save_model("layoutlmv3-fdr")
processor.save_pretrained("layoutlmv3-fdr")
json.dump({"data": "synthetic", "entity_f1": result.get("eval_entity_f1"), "train_pages": N_TRAIN,
           "test_pages": N_TEST, "epochs": EPOCHS, "learning_rate": LEARNING_RATE},
          open("layoutlmv3-fdr/metrics.json", "w"), indent=2)
shutil.make_archive("layoutlmv3-fdr", "zip", "layoutlmv3-fdr")
try:
    from google.colab import files
    files.download("layoutlmv3-fdr.zip")
except ImportError:
    print("Download layoutlmv3-fdr.zip from the file browser.")